# Notebook 1: Data Merging & Cleaning
**Project: Analisis Komentar YouTube #PESTABABI**

Tujuan notebook ini:
1. Merge dataset `Daftar_Video` dan `Data_Mentah` menggunakan `Video_ID` sebagai kunci
2. Handling missing values & duplicates
3. Perbaikan tipe data (datetime, integer)
4. Simpan hasil ke `data/processed/`

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import os

# Pastikan path relatif ke root project
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')

print(f'Base directory : {BASE_DIR}')
print(f'Raw data       : {RAW_DIR}')
print(f'Processed data : {PROCESSED_DIR}')

Base directory : c:\Users\hp\Downloads\pestababi_project\pestababi_project
Raw data       : c:\Users\hp\Downloads\pestababi_project\pestababi_project\data\raw
Processed data : c:\Users\hp\Downloads\pestababi_project\pestababi_project\data\processed


## 2. Load Raw Data

In [2]:
df_video = pd.read_csv(os.path.join(RAW_DIR, 'PESTABABI_-_Daftar_Video.csv'))
df_komentar = pd.read_csv(os.path.join(RAW_DIR, 'PESTABABI_-_Data_Mentah.csv'))

print('=== Daftar Video ===')
print(f'Shape: {df_video.shape}')
display(df_video.head(3))

print('\n=== Data Mentah (Komentar) ===')
print(f'Shape: {df_komentar.shape}')
display(df_komentar.head(3))

=== Daftar Video ===
Shape: (25, 4)


,Video_ID,Judul_Video,Channel,Tanggal_Rilis
0,Kjq6YuxPWuw,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17T18:37:03Z
1,PjeNsvCSmTY,PESTA BABI Full Movie (Dokumenter) Bebas Iklan,Melanesia Movies,2026-05-18T08:54:18Z
2,lslu0_8OynQ,Represi dalam Pemutaran Film Pesta Babi | Boco...,Tempodotco,2026-05-16T04:00:06Z



=== Data Mentah (Komentar) ===
Shape: (10945, 3)


,Video_ID,Komentar,Jumlah_Suka
0,Kjq6YuxPWuw,"Tolong dihapus videonya, ini pelanggaran Hak C...",7
1,Kjq6YuxPWuw,😢😢😢,1
2,Kjq6YuxPWuw,Begitu kejam nya pemerintah terhadap rakyatnya...,1


## 3. Merge Dataset

Kita gunakan **left join** — semua komentar dipertahankan.
Komentar yang tidak memiliki pasangan video (jika ada) tetap masuk, tapi akan difilter pada tahap cleaning.

In [3]:
df = pd.merge(
    left=df_komentar,
    right=df_video,
    on='Video_ID',
    how='left'
)

print(f'Shape setelah merge: {df.shape}')
print(f'Kolom: {df.columns.tolist()}')
display(df.head(5))

Shape setelah merge: (10945, 6)
Kolom: ['Video_ID', 'Komentar', 'Jumlah_Suka', 'Judul_Video', 'Channel', 'Tanggal_Rilis']


,Video_ID,Komentar,Jumlah_Suka,Judul_Video,Channel,Tanggal_Rilis
0,Kjq6YuxPWuw,"Tolong dihapus videonya, ini pelanggaran Hak C...",7,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17T18:37:03Z
1,Kjq6YuxPWuw,😢😢😢,1,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17T18:37:03Z
2,Kjq6YuxPWuw,Begitu kejam nya pemerintah terhadap rakyatnya...,1,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17T18:37:03Z
3,Kjq6YuxPWuw,Mengerikan.....😢😢😢,4,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17T18:37:03Z
4,PjeNsvCSmTY,Film Keren Abad ini,0,PESTA BABI Full Movie (Dokumenter) Bebas Iklan,Melanesia Movies,2026-05-18T08:54:18Z


## 4. Inspeksi Awal

In [4]:
print('=== Info DataFrame ===')
df.info()

print('\n=== Missing Values per Kolom ===')
print(df.isnull().sum())

print(f'\n=== Jumlah Baris Duplikat: {df.duplicated().sum()} ===')

=== Info DataFrame ===
<class 'pandas.DataFrame'>
RangeIndex: 10945 entries, 0 to 10944
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   Video_ID       10945 non-null  str  
 1   Komentar       10945 non-null  str  
 2   Jumlah_Suka    10945 non-null  int64
 3   Judul_Video    10945 non-null  str  
 4   Channel        10945 non-null  str  
 5   Tanggal_Rilis  10945 non-null  str  
dtypes: int64(1), str(5)
memory usage: 513.2 KB

=== Missing Values per Kolom ===
Video_ID         0
Komentar         0
Jumlah_Suka      0
Judul_Video      0
Channel          0
Tanggal_Rilis    0
dtype: int64

=== Jumlah Baris Duplikat: 154 ===


## 5. Handling Missing Values

In [5]:
rows_before = len(df)

# Hapus baris di mana kolom krusial ini kosong
df.dropna(subset=['Komentar', 'Video_ID', 'Judul_Video', 'Channel'], inplace=True)

# Hapus komentar yang hanya berisi whitespace/spasi
df = df[df['Komentar'].str.strip() != '']

rows_after = len(df)
print(f'Baris sebelum: {rows_before}')
print(f'Baris setelah hapus missing: {rows_after}')
print(f'Baris dihapus: {rows_before - rows_after}')

Baris sebelum: 10945
Baris setelah hapus missing: 10945
Baris dihapus: 0


## 6. Handling Duplicates

In [6]:
rows_before = len(df)

# Duplikat = baris dengan Komentar DAN Video_ID yang sama persis
df.drop_duplicates(subset=['Video_ID', 'Komentar'], keep='first', inplace=True)

rows_after = len(df)
print(f'Baris sebelum: {rows_before}')
print(f'Baris setelah hapus duplikat: {rows_after}')
print(f'Duplikat dihapus: {rows_before - rows_after}')

Baris sebelum: 10945
Baris setelah hapus duplikat: 10763
Duplikat dihapus: 182


## 7. Data Type Fixing

In [7]:
# Konversi Tanggal_Rilis ke datetime (format ISO 8601 dari YouTube API)
df['Tanggal_Rilis'] = pd.to_datetime(df['Tanggal_Rilis'], errors='coerce', utc=True)

# Konversi Jumlah_Suka ke integer (isi NaN dengan 0 terlebih dahulu)
df['Jumlah_Suka'] = df['Jumlah_Suka'].fillna(0).astype(int)

# Reset index setelah semua operasi cleaning
df.reset_index(drop=True, inplace=True)

print('=== Tipe Data Setelah Fix ===')
print(df.dtypes)

print('\n=== Sample Data Bersih ===')
display(df.head(5))

=== Tipe Data Setelah Fix ===
Video_ID                         str
Komentar                         str
Jumlah_Suka                    int64
Judul_Video                      str
Channel                          str
Tanggal_Rilis    datetime64[us, UTC]
dtype: object

=== Sample Data Bersih ===


,Video_ID,Komentar,Jumlah_Suka,Judul_Video,Channel,Tanggal_Rilis
0,Kjq6YuxPWuw,"Tolong dihapus videonya, ini pelanggaran Hak C...",7,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17 18:37:03+00:00
1,Kjq6YuxPWuw,😢😢😢,1,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17 18:37:03+00:00
2,Kjq6YuxPWuw,Begitu kejam nya pemerintah terhadap rakyatnya...,1,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17 18:37:03+00:00
3,Kjq6YuxPWuw,Mengerikan.....😢😢😢,4,Pesta babi (Official trailer full movie),Gilberth official,2026-05-17 18:37:03+00:00
4,PjeNsvCSmTY,Film Keren Abad ini,0,PESTA BABI Full Movie (Dokumenter) Bebas Iklan,Melanesia Movies,2026-05-18 08:54:18+00:00


## 8. Tambah Kolom Bantu (Feature Engineering Dasar)

In [8]:
# Kolom panjang karakter komentar — berguna untuk EDA & filtering
df['Panjang_Komentar'] = df['Komentar'].str.len()

# Kolom jumlah kata dalam komentar
df['Jumlah_Kata'] = df['Komentar'].str.split().str.len()

# Kategori likes: 0, low (1-4), medium (5-19), high (>=20)
def kategorikan_likes(likes):
    if likes == 0:
        return 'none'
    elif likes < 5:
        return 'low'
    elif likes < 20:
        return 'medium'
    else:
        return 'high'

df['Kategori_Likes'] = df['Jumlah_Suka'].apply(kategorikan_likes)

print('Kolom baru ditambahkan:')
print(df[['Komentar', 'Jumlah_Suka', 'Panjang_Komentar', 'Jumlah_Kata', 'Kategori_Likes']].head(5).to_string())

Kolom baru ditambahkan:
                                                                                                              Komentar  Jumlah_Suka  Panjang_Komentar  Jumlah_Kata Kategori_Likes
0  Tolong dihapus videonya, ini pelanggaran Hak Cipta. Karena pihak Watchdoc belum keluarkan film ini akun resminya 🙏🏽            7               115           17         medium
1                                                                                                                  😢😢😢            1                 3            1            low
2                                                           Begitu kejam nya pemerintah terhadap rakyatnya sendiri.😢😢😢            1                58            7            low
3                                                                                                   Mengerikan.....😢😢😢            4                18            1            low
4                                                                                     

## 9. Simpan Data Bersih

In [9]:
os.makedirs(PROCESSED_DIR, exist_ok=True)

output_path = os.path.join(PROCESSED_DIR, 'pestababi_clean.csv')
df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ Data bersih tersimpan di: {output_path}')
print(f'   Total baris final: {len(df)}')
print(f'   Total kolom final: {len(df.columns)}')
print(f'   Kolom: {df.columns.tolist()}')

✅ Data bersih tersimpan di: c:\Users\hp\Downloads\pestababi_project\pestababi_project\data\processed\pestababi_clean.csv
   Total baris final: 10763
   Total kolom final: 9
   Kolom: ['Video_ID', 'Komentar', 'Jumlah_Suka', 'Judul_Video', 'Channel', 'Tanggal_Rilis', 'Panjang_Komentar', 'Jumlah_Kata', 'Kategori_Likes']
